# Means, medians, and trimmed medians of annual tide-gauge trends

This notebook implements the observational summaries in *From Local Tide-Gauge Trends to Sea-Level Rise Estimates: Means, Medians, and Trimming*.

The calculation uses PSMSL annual RLR records, sliding **60- and 70-year windows within 1900–2019**, minimum completeness of **75% and 87%**, and statistically significant positive OLS slopes. It calculates the arithmetic mean, median, and upper-tail-trimmed estimator, with the associated station-subsampling dispersion. All functions are included; no previous notebook or live kernel is required.

**Execution:** set `DATA_DIR` in Section 1, run the definition and check cells, inspect the final-window pilot in Section 6, then set `RUN_FULL = True` in Section 7. Section 7 is the main computation. Sections 8–12 inspect results, create figures, and export tables and provenance. The default is 1,000 subsamples per window; there is no outer loop of 100 or 200 synthetic realizations.

**Data snapshot:** reproduction of historical station counts requires the same annual-data snapshot as the manuscript. The 2019 endpoint applies to the fitted windows. A later PSMSL download may also revise earlier observations. The notebook records input-file hashes and package versions.

**Implementation conventions:** nearest-integer rounding with ties upward is used for retained station counts and half-sample sizes, as in the supplied MATLAB trimming code. Subsets are independent across repetitions and contain no repeated station within a draw. The legacy MATLAB helper attempted to reject repeated subsets across draws; the present implementation uses uniform independent subsampling, consistent with the revision's Python sensitivity analyses. Archived and newly generated Monte Carlo summaries can consequently differ slightly. Direct means, medians, selected station identities, and fitted slopes are deterministic for a fixed dataset and configuration.

The notebook implements the baseline **OLS** selection. HAC/BH, fixed-cohort, and stochastic-model sensitivity analyses remain separate calculations.


## 1. Dependencies and settings

Dependencies: Python 3.10 or later, NumPy, SciPy, pandas, Matplotlib, IPython, and Jupyter. If needed, install the scientific packages in the notebook environment with `%pip install numpy scipy pandas matplotlib ipython`. Package versions are recorded in the exports.

`DATA_DIR` identifies the extracted annual RLR folder containing `filelist.txt` and `data/*.rlrdata`. The required numerical threshold is **two-sided $p\leq0.01$**, equivalent to **one-sided $p\leq0.005$ for positive fitted slopes**. The defaults reproduce the three trimming fractions displayed in the observational figures: $\gamma=0,0.10,0.25$. Additional fractions can be added to `gammas`.

`n_resamples=1000` retains the supplied envelope construction. A smaller value may be used for exploratory timing, in which case the nominal 99% envelope is left undefined. The full-run outputs use a frozen copy of the settings to prevent later cell edits from relabeling existing calculations.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import os
import glob
import json
import hashlib
import platform
import time
import numpy as np
import scipy
from scipy import stats
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display

DATA_DIR = Path('../data/rlr_annual')
OUTPUT_DIR = Path('annual_tide_gauge_results')
DATA_SNAPSHOT_NOTE = 'Annual PSMSL RLR snapshot used in the manuscript; add access date or archive identifier.'

CONFIG = dict(
    year_limits=[1900, 2019],
    window_lengths=[60, 70],
    completeness_values=[0.75, 0.87],
    pcrit_two_sided=0.01,
    gammas=[0.0, 0.10, 0.25],
    subsample_fraction=0.5,
    n_resamples=1000,
    seed=20261126,
)

print('Data directory:', DATA_DIR.resolve())
print('Output directory:', OUTPUT_DIR.resolve())
print(json.dumps(CONFIG, indent=2))


Data directory: C:\Users\User\work\IPGP\Climate\data\rlr_annual
Output directory: C:\Users\User\work\IPGP\Climate\bin\annual_tide_gauge_results
{
  "year_limits": [
    1900,
    2019
  ],
  "window_lengths": [
    60,
    70
  ],
  "completeness_values": [
    0.75,
    0.87
  ],
  "pcrit_two_sided": 0.01,
  "gammas": [
    0.0,
    0.1,
    0.25
  ],
  "subsample_fraction": 0.5,
  "n_resamples": 1000,
  "seed": 20261126
}


## 2. Data handling and station selection

A window starting at $y$ contains the $w$ annual observations indexed by $y,\ldots,y+w-1$. The figure coordinate is the legacy label $y+w/2$; both exact interval bounds are exported. Thus the 1950–2019 window is labeled 1985.

For each station:

1. The first and last years recorded in its input file must span the full window.
2. An explicit calendar grid represents absent years, nonfinite observations, and the sentinel `-99999` as missing. Values are not interpolated. Missing entries at window boundaries count toward incompleteness.
3. At least the prescribed fraction of years must have valid observations. A linear trend with an intercept is fitted against actual calendar years.
4. The station enters the selected set if its slope is positive and its two-sided OLS p-value is at most 0.01.

The reader retains the existing flag policy: flags are parsed, with no additional exclusions based on station or annual-record flags. The same validity mask is used for completeness and regression. The station-fit export includes all windows meeting the least restrictive configured completeness threshold; a stricter threshold is applied to those fits without refitting.

The following cells define the data reader, OLS calculation, and window selection.


In [2]:
import os
import glob
import numpy as np


def read_annual(directory_name):
    """
    Read a PSMSL RLR annual data directory.
    
    Parameters:
        directory_name (str): Path to unzipped PSMSL directory containing 
                              'filelist.txt' and the 'data/' folder.
                              
    Returns:
        list of dict: Each dict corresponds to a station with keys:
            'id', 'latitude', 'longitude', 'name', 'coastline',
            'stationcode', 'stationflag', 'year', 'height',
            'interpolated', 'isMtl', 'dataflag'
    """
    if not os.path.isdir(directory_name):
        raise FileNotFoundError(f"Cannot find directory: {directory_name}")

    data_dir = os.path.join(directory_name, 'data')
    if not os.path.isdir(data_dir):
        raise FileNotFoundError(
            f"Cannot find data directory {data_dir}. "
            "The path must identify the root extracted folder containing data/."
        )

    # 1. Parse filelist.txt catalogue
    cat_file = os.path.join(directory_name, 'filelist.txt')
    if not os.path.isfile(cat_file):
        raise FileNotFoundError(f"Could not find catalogue file: {cat_file}")

    catalogue = {}
    with open(cat_file, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(';')
            if len(parts) < 7:
                continue
            st_id = int(parts[0].strip())
            catalogue[st_id] = {
                'latitude': float(parts[1].strip()),
                'longitude': float(parts[2].strip()),
                'name': parts[3].strip(),
                'coastline': int(parts[4].strip()) if parts[4].strip() else None,
                'stationcode': int(parts[5].strip()) if parts[5].strip() else None,
                'stationflag': parts[6].strip() == 'Y'
            }

    # 2. Match .rlrdata files
    file_paths = glob.glob(os.path.join(data_dir, '*.rlrdata'))
    data = []

    for path in sorted(file_paths):
        filename = os.path.basename(path)
        st_id = int(filename.split('.')[0])
        
        meta = catalogue.get(st_id, {
            'latitude': np.nan, 'longitude': np.nan, 'name': '',
            'coastline': None, 'stationcode': None, 'stationflag': False
        })

        # 3. Read station time series
        # Format: Year; Height; InterpolatedFlag; QualityFlags(3 digits)
        years, heights, interpolated, is_mtl, dataflag = [], [], [], [], []
        
        with open(path, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split(';')
                if len(parts) < 4:
                    continue

                yr = int(parts[0].strip())
                raw_h = float(parts[1].strip())
                h = np.nan if raw_h == -99999 else raw_h
                interp = parts[2].strip() == 'Y'

                # Flags: 3-character string (e.g. "000")
                flags_str = parts[3].strip()
                flags_str = flags_str.ljust(3, '0')
                mtl = flags_str[1] == '1'
                dflag = flags_str[2] == '1'

                years.append(yr)
                heights.append(h)
                interpolated.append(interp)
                is_mtl.append(mtl)
                dataflag.append(dflag)

        station_record = {
            'id': st_id,
            'latitude': meta['latitude'],
            'longitude': meta['longitude'],
            'name': meta['name'],
            'coastline': meta['coastline'],
            'stationcode': meta['stationcode'],
            'stationflag': meta['stationflag'],
            'year': np.array(years, dtype=int),
            'height': np.array(heights, dtype=float),
            'interpolated': np.array(interpolated, dtype=bool),
            'isMtl': np.array(is_mtl, dtype=bool),
            'dataflag': np.array(dataflag, dtype=bool)
        }
        data.append(station_record)

    return data


In [3]:
def fit_linear_ols(x, y):
    """
    Computes analytical OLS slope and two-tailed p-value.
    Use a Student t reference with n-2 residual degrees of freedom.
    """
    n = len(x)
    if n < 3:
        return 0.0, 1.0

    x_mean = np.mean(x)
    y_mean = np.mean(y)
    x_dev = x - x_mean
    ss_xx = np.sum(x_dev**2)

    if ss_xx == 0.0:
        return 0.0, 1.0

    slope = np.sum(x_dev * (y - y_mean)) / ss_xx
    intercept = y_mean - slope * x_mean

    residuals = y - (intercept + slope * x)
    df = n - 2
    s_err2 = np.sum(residuals**2) / df
    se_slope = np.sqrt(s_err2 / ss_xx)

    if se_slope == 0.0:
        return slope, 0.0

    t_stat = slope / se_slope
    p_val = 2.0 * stats.t.sf(np.abs(t_stat), df=df)
    return slope, p_val


In [4]:
def validate_config(cfg):
    first, last = cfg['year_limits']
    if int(first) != first or int(last) != last or first > last:
        raise ValueError('year_limits must contain ordered integer years.')
    for key in ('window_lengths', 'completeness_values', 'gammas'):
        if not cfg[key] or len(set(cfg[key])) != len(cfg[key]):
            raise ValueError(f'{key} must contain distinct values.')
    if any(int(w) != w or not 3 <= w <= last-first+1 for w in cfg['window_lengths']):
        raise ValueError('Invalid window length.')
    if any(not 0 < f <= 1 for f in cfg['completeness_values']):
        raise ValueError('Completeness must be in (0, 1].')
    if any(not 0 <= g < 1 for g in cfg['gammas']) or 0.0 not in cfg['gammas']:
        raise ValueError('gammas must include 0 and lie in [0, 1).')
    if not 0 < cfg['pcrit_two_sided'] < 1:
        raise ValueError('Invalid significance threshold.')
    if not 0 < cfg['subsample_fraction'] <= 1:
        raise ValueError('Invalid subsample fraction.')
    if int(cfg['n_resamples']) != cfg['n_resamples'] or cfg['n_resamples'] < 2:
        raise ValueError('n_resamples must be an integer of at least 2.')
    if int(cfg['seed']) != cfg['seed'] or cfg['seed'] < 0:
        raise ValueError('seed must be a nonnegative integer.')


def round_half_up(value):
    """Nearest integer, with positive half-integers rounded upward."""
    return int(np.floor(float(value) + 0.5))


def prepare_records(series, cfg):
    """Create a calendar grid while retaining each raw record's full span."""
    validate_config(cfg)
    first, last = cfg['year_limits']
    records, seen = [], set()
    for st in series:
        sid = int(st['id'])
        if sid in seen:
            raise ValueError(f'Duplicate station ID: {sid}')
        seen.add(sid)
        years = np.asarray(st['year'], dtype=float)
        heights = np.asarray(st['height'], dtype=float)
        if years.ndim != 1 or heights.shape != years.shape:
            raise ValueError(f'Invalid year/height arrays: station {sid}.')
        if not len(years):
            continue
        if (not np.isfinite(years).all() or np.any(years != np.rint(years))
                or np.any(np.diff(years) <= 0)):
            raise ValueError(f'Annual years must be unique increasing integers: station {sid}.')
        years = years.astype(int)
        grid = np.full(last-first+1, np.nan)
        inside = (years >= first) & (years <= last)
        grid[years[inside]-first] = heights[inside]
        grid[(grid == -99999) | ~np.isfinite(grid)] = np.nan
        records.append(dict(station_id=sid, name=st.get('name', ''),
                            latitude=st.get('latitude', np.nan),
                            longitude=st.get('longitude', np.nan),
                            record_start=int(years[0]), record_end=int(years[-1]),
                            values=grid))
    return records


FIT_COLUMNS = ['window_length', 'start_year', 'end_year', 'window_label',
               'station_id', 'name', 'latitude', 'longitude', 'n_obs',
               'n_missing', 'observed_fraction', 'slope', 'p_two_sided',
               'p_one_sided', 'positive_significant']


def fit_one_window(records, start, length, cfg):
    """Fit records meeting the least restrictive completeness setting."""
    end = start + length - 1
    offset = start - cfg['year_limits'][0]
    if offset < 0 or end > cfg['year_limits'][1]:
        raise ValueError('Window falls outside the configured analysis period.')
    years = np.arange(start, end+1)
    rows = []
    for st in records:
        if st['record_start'] > start or st['record_end'] < end:
            continue
        chunk = st['values'][offset:offset+length]
        valid = np.isfinite(chunk)
        n_obs = int(valid.sum())
        if n_obs < 3 or n_obs/length < min(cfg['completeness_values']):
            continue
        slope, p_two = fit_linear_ols(years[valid], chunk[valid])
        p_one = p_two/2 if slope > 0 else (1-p_two/2 if slope < 0 else 0.5)
        rows.append(dict(window_length=length, start_year=start, end_year=end,
                         window_label=start+length/2, station_id=st['station_id'],
                         name=st['name'], latitude=st['latitude'], longitude=st['longitude'],
                         n_obs=n_obs, n_missing=length-n_obs, observed_fraction=n_obs/length,
                         slope=float(slope), p_two_sided=float(p_two), p_one_sided=float(p_one),
                         positive_significant=bool(slope > 0 and p_two <= cfg['pcrit_two_sided'])))
    return pd.DataFrame(rows, columns=FIT_COLUMNS)


def fit_windows(records, cfg, final_only=False, progress=True):
    """Fit each station-window once and reuse it across completeness settings."""
    validate_config(cfg)
    first, last = cfg['year_limits']
    frames = []
    for w in cfg['window_lengths']:
        starts = [last-w+1] if final_only else range(first, last-w+2)
        for i, start in enumerate(starts):
            frames.append(fit_one_window(records, start, w, cfg))
            if progress and (final_only or i == 0 or (i+1) % 10 == 0 or start == last-w+1):
                print(f'Fitted {w}-year window {start}–{start+w-1}', flush=True)
    return pd.concat(frames, ignore_index=True)


## 3. Estimators, subsampling, and uncertainty notation

Let $X$ contain the $N$ selected local slopes in one window. The direct summaries are its arithmetic mean and median. Upper-tail trimming retains the smallest

$$N_\gamma=\lfloor(1-\gamma)N+1/2\rfloor$$

slopes. Each draw then contains

$$m_\gamma=\lfloor fN_\gamma+1/2\rfloor,\qquad f=0.5,$$

stations selected uniformly **without replacement**. At an exact tie at the trimming boundary, station ID supplies a deterministic ordering; the retained numerical slopes are unchanged. Separate repetitions may contain identical subsets.

For $B=1000$ draws, the notebook records three different resampled summaries:

| Output column | Calculation | Use |
|---|---|---|
| `resampled_mean` | Mean of the $B$ subsample means | Mean comparison at $\gamma=0$ |
| `resampled_median_mean` | Mean of the $B$ subsample medians | Median comparison at $\gamma=0$ |
| `M_prime` | Median of the $B$ subsample medians | Final estimator at each $\gamma$ |

`mean_selected` and `median_selected` are the direct summaries before trimming; `median_retained` is the direct median after trimming. Finite subsampling can make `M_prime` differ from `median_retained`.

The SD of the subsample means or medians uses denominator $B-1$. For subsample means, an exact finite-population comparison is also exported:

$$\operatorname{SD}(\overline{w}_m\mid X)=\sqrt{\frac{1-m/N}{m}\,S^2},$$

where $S^2$ uses denominator $N-1$. This formula is evaluated on the retained population after trimming.

**Nominal 99% envelope for $M'$.** The original trajectory calculation is retained explicitly. With window index $j$, subsample median $Q_{jb}$, and $M'_j=\operatorname{Med}_b Q_{jb}$, define

$$D_b=\sqrt{\sum_j(Q_{jb}-M'_j)^2},\qquad
S_j=\frac{s_j}{\sqrt{\sum_k s_k^2}}D_{(991)},$$

where $s_j$ is the subsample-median SD and $D_{(991)}$ is the 991st ordered value among 1000 norms. The resulting display is $M'_j\pm S_j$. The envelope is calculated separately for each window length, completeness threshold, and trimming fraction over all windows with defined estimates.

This is a **descriptive subsampling envelope conditional on the selected and retained stations**. Subsamples are drawn separately within each window; combining their repetition indices does not preserve covariance between overlapping windows or establish simultaneous 99% coverage. The envelope and SDs describe the computational station summary, rather than the uncertainty of a geographically weighted global trend. The final-window pilot leaves trajectory envelopes undefined because it omits the other windows.


In [5]:
def resample_slopes(slopes, gamma, sample_fraction, repeats, rng):
    """Trim largest slopes, then sample stations independently without replacement."""
    x = np.asarray(slopes, dtype=float)
    if x.ndim != 1 or not np.isfinite(x).all():
        raise ValueError('Slopes must be a finite one-dimensional array.')
    if not 0 <= gamma < 1 or not 0 < sample_fraction <= 1:
        raise ValueError('Invalid trimming or subsample fraction.')
    if int(repeats) != repeats or repeats < 2:
        raise ValueError('At least two integer repetitions are required.')
    x = np.sort(x, kind='stable')
    n_retained = round_half_up((1-gamma)*len(x))
    m = round_half_up(sample_fraction*n_retained)
    retained = x[:n_retained]
    means = np.full(repeats, np.nan)
    medians = np.full(repeats, np.nan)
    if n_retained and m:
        for b in range(repeats):
            indices = rng.choice(n_retained, size=m, replace=False)
            sample = retained[indices]
            means[b] = np.mean(sample)
            medians[b] = np.median(sample)
    return dict(n_retained=n_retained, sample_size=m, retained_slopes=retained,
                removed_slopes=x[n_retained:], means=means, medians=medians)


def conditional_mean_sd(slopes, sample_size):
    """Exact conditional SD for a uniform sample mean without replacement."""
    x = np.asarray(slopes, dtype=float)
    n, m = len(x), int(sample_size)
    if not n or not m:
        return np.nan
    if not 1 <= m <= n:
        raise ValueError('Sample size must lie between 1 and the population size.')
    if n == 1 or m == n:
        return 0.0
    return float(np.sqrt((1-m/n)*np.var(x, ddof=1)/m))


def numeric_seed_key(value):
    """Stable key for configuration decimals, independent of Python hash state."""
    token = format(float(value), '.12g').encode('ascii')
    return int.from_bytes(hashlib.sha256(token).digest()[:4], 'little')


def result_rng(cfg, start, length, completeness, gamma):
    key = [cfg['seed'], start, length, numeric_seed_key(completeness), numeric_seed_key(gamma)]
    return np.random.default_rng(np.random.SeedSequence(key))


def trajectory_envelope(medians, centers):
    """Legacy nominal 99% norm envelope, defined only for B=1000."""
    draws = np.asarray(medians, dtype=float)
    centers = np.asarray(centers, dtype=float)
    if draws.ndim != 2 or centers.shape != (draws.shape[0],):
        raise ValueError('Expected window-by-repetition draws and one center per window.')
    if draws.shape[1] != 1000:
        raise ValueError('The legacy 991st-order-statistic envelope requires 1000 repetitions.')
    valid = np.isfinite(draws).all(axis=1) & np.isfinite(centers)
    band = np.full(len(centers), np.nan)
    if not valid.any():
        return band, np.nan, 0
    sd = np.std(draws[valid], axis=1, ddof=1)
    norm = np.sqrt(np.sum((draws[valid]-centers[valid, None])**2, axis=0))
    q99 = float(np.sort(norm)[990])
    denominator = np.linalg.norm(sd)
    band[valid] = 0.0 if denominator == 0 else sd/denominator*q99
    return band, q99, int(valid.sum())


In [6]:
def estimate_windows(fits, cfg, final_only=False, progress=True):
    """Return summaries, resampling arrays, and the station-level trimming audit."""
    validate_config(cfg)
    first, last = cfg['year_limits']
    by_window = {key: frame for key, frame in fits.groupby(['window_length', 'start_year'])}
    empty = pd.DataFrame(columns=FIT_COLUMNS)
    rows, membership, draws = [], [], {}
    for w in cfg['window_lengths']:
        starts = [last-w+1] if final_only else list(range(first, last-w+2))
        for fraction in cfg['completeness_values']:
            for gamma in cfg['gammas']:
                key = (int(w), float(fraction), float(gamma))
                draws[key] = dict(start_year=np.asarray(starts, int),
                                  means=np.full((len(starts), cfg['n_resamples']), np.nan),
                                  medians=np.full((len(starts), cfg['n_resamples']), np.nan))
            for j, start in enumerate(starts):
                frame = by_window.get((w, start), empty)
                eligible = frame.loc[frame['observed_fraction'] >= fraction]
                chosen = eligible.loc[eligible['positive_significant'] == True].sort_values(
                    ['slope', 'station_id'], kind='stable')
                x = chosen['slope'].to_numpy(float)
                for gamma in cfg['gammas']:
                    sample = resample_slopes(x, gamma, cfg['subsample_fraction'],
                                             cfg['n_resamples'], result_rng(cfg, start, w, fraction, gamma))
                    key = (int(w), float(fraction), float(gamma))
                    draws[key]['means'][j] = sample['means']
                    draws[key]['medians'][j] = sample['medians']
                    defined = sample['sample_size'] > 0
                    retained = sample['retained_slopes']
                    common = dict(window_length=w, completeness=float(fraction),
                                  alpha=float(round(1-fraction, 12)), gamma=float(gamma),
                                  start_year=start, end_year=start+w-1, window_label=start+w/2)
                    rows.append(dict(common, eligible=len(eligible), selected=len(x),
                        n_retained=sample['n_retained'], sample_size=sample['sample_size'],
                        mean_selected=float(np.mean(x)) if len(x) else np.nan,
                        median_selected=float(np.median(x)) if len(x) else np.nan,
                        mean_retained=float(np.mean(retained)) if len(retained) else np.nan,
                        median_retained=float(np.median(retained)) if len(retained) else np.nan,
                        resampled_mean=float(np.mean(sample['means'])) if defined else np.nan,
                        resampled_median_mean=float(np.mean(sample['medians'])) if defined else np.nan,
                        M_prime=float(np.median(sample['medians'])) if defined else np.nan,
                        mean_subsample_sd=float(np.std(sample['means'], ddof=1)) if defined else np.nan,
                        median_subsample_sd=float(np.std(sample['medians'], ddof=1)) if defined else np.nan,
                        mean_subsample_sd_exact=conditional_mean_sd(retained, sample['sample_size']),
                        nominal_99_halfwidth=np.nan, norm_order_statistic_991=np.nan,
                        envelope_valid_windows=0))
                    for rank, st in enumerate(chosen.itertuples(index=False), 1):
                        membership.append(dict(common, station_id=int(st.station_id), name=st.name,
                                               slope=float(st.slope), rank=rank,
                                               retained=bool(rank <= sample['n_retained'])))
            if progress:
                print(f'Estimated {w}-year windows; completeness={fraction:.2f}', flush=True)
    summary = pd.DataFrame(rows)
    if not final_only and cfg['n_resamples'] == 1000:
        for (w, fraction, gamma), arrays in draws.items():
            mask = ((summary.window_length == w) & (summary.completeness == fraction)
                    & (summary.gamma == gamma))
            centers = summary.loc[mask, 'M_prime'].to_numpy(float)
            band, q99, n_valid = trajectory_envelope(arrays['medians'], centers)
            summary.loc[mask, 'nominal_99_halfwidth'] = band
            summary.loc[mask, 'norm_order_statistic_991'] = q99
            summary.loc[mask, 'envelope_valid_windows'] = n_valid
    members = pd.DataFrame(membership, columns=[
        'window_length', 'completeness', 'alpha', 'gamma', 'start_year', 'end_year',
        'window_label', 'station_id', 'name', 'slope', 'rank', 'retained'])
    return summary, draws, members


def input_manifest(data_dir):
    """Record hashes of the catalogue and every input annual-record file."""
    root = Path(data_dir)
    paths = [root/'filelist.txt'] + sorted((root/'data').glob('*.rlrdata'))
    entries = [dict(path=str(p.relative_to(root)), size_bytes=p.stat().st_size,
                    sha256=hashlib.sha256(p.read_bytes()).hexdigest()) for p in paths]
    digest = hashlib.sha256(json.dumps(entries, sort_keys=True).encode('utf-8')).hexdigest()
    return entries, digest


def assert_results(summary, cfg):
    """Check aggregation identities, counts, and the finite-population reference."""
    assert (summary.selected <= summary.eligible).all()
    assert (summary.n_retained <= summary.selected).all()
    assert (summary.sample_size <= summary.n_retained).all()
    assert (summary.n_retained == np.floor((1-summary.gamma)*summary.selected + 0.5)).all()
    assert (summary.sample_size == np.floor(cfg['subsample_fraction']*summary.n_retained + 0.5)).all()
    finite = summary.M_prime.notna()
    assert (summary.loc[finite, 'M_prime'] > 0).all()
    for _, group in summary.groupby(['window_length', 'completeness', 'start_year']):
        g = group.sort_values('gamma')
        r = g.median_retained.dropna().to_numpy()
        assert np.all(np.diff(r) <= 1e-12)
    return True


## 4. Small numerical checks (no observational data required)

These checks cover rounding, the trimming rule, full-sample behavior, reproducible subsampling, the exact finite-population variance identity, and the distinction between averaging and taking the median of subsample medians. They are computational checks only.


In [7]:
from itertools import combinations

validate_config(CONFIG)
assert round_half_up(37.5) == 38
assert round_half_up(67.5) == 68

x_check = np.array([1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 6.0, 10.0])
full_check = resample_slopes(x_check, gamma=0.25, sample_fraction=1.0,
                           repeats=1000, rng=np.random.default_rng(1))
assert full_check['n_retained'] == full_check['sample_size'] == 6
np.testing.assert_array_equal(full_check['removed_slopes'], [6.0, 10.0])
np.testing.assert_allclose(full_check['medians'], 1.5)
np.testing.assert_allclose(full_check['means'], 1.5)

half_check = resample_slopes(x_check, gamma=0.25, sample_fraction=0.5,
                           repeats=1000, rng=np.random.default_rng(1))
assert half_check['sample_size'] == 3
repeat_check = resample_slopes(x_check, gamma=0.25, sample_fraction=0.5,
                             repeats=1000, rng=np.random.default_rng(1))
np.testing.assert_array_equal(half_check['medians'], repeat_check['medians'])

population = np.array([0.0, 1.0, 3.0, 9.0])
all_means = np.array([population[list(ids)].mean() for ids in combinations(range(4), 2)])
np.testing.assert_allclose(all_means.mean(), population.mean())
np.testing.assert_allclose(all_means.std(ddof=0), conditional_mean_sd(population, 2))
assert conditional_mean_sd(population, 4) == 0.0
assert conditional_mean_sd([2.0], 1) == 0.0

illustrative_medians = np.array([1.0, 1.0, 4.0])
assert illustrative_medians.mean() == 2.0
assert np.median(illustrative_medians) == 1.0
empty_check = resample_slopes([], 0.25, 0.5, 1000, np.random.default_rng(0))
assert empty_check['sample_size'] == 0 and np.isnan(empty_check['medians']).all()
flat_band, _, _ = trajectory_envelope(np.ones((3, 1000)), np.ones(3))
np.testing.assert_array_equal(flat_band, np.zeros(3))
print('Estimator checks passed.')


Estimator checks passed.


## 5. Read the annual records

The input manifest fingerprints the catalogue and all annual station files. `DATA_SNAPSHOT_NOTE` can identify the access date or archived dataset. Original input files remain unchanged.


In [8]:
manifest_before, fingerprint_before = input_manifest(DATA_DIR)
series = read_annual(str(DATA_DIR))
if not series:
    raise ValueError('No annual station records were found.')
INPUT_MANIFEST, INPUT_FINGERPRINT = input_manifest(DATA_DIR)
if INPUT_FINGERPRINT != fingerprint_before:
    raise RuntimeError('Input files changed during loading; reload the records.')
records = prepare_records(series, CONFIG)
LOADED_YEAR_LIMITS = list(CONFIG['year_limits'])
print('Catalogue station records:', len(series))
print('Prepared nonempty station records:', len(records))
print('Input manifest SHA-256:', INPUT_FINGERPRINT)
display(pd.DataFrame([dict(station_id=s['station_id'], name=s['name'],
                          record_start=s['record_start'], record_end=s['record_end'])
                      for s in records]).head(8))


Catalogue station records: 1548
Prepared nonempty station records: 1533
Input manifest SHA-256: 807d045f2c93f86359476fa732cb8a4c1a93dd3ba07d4a6128204314af09e084


,station_id,name,record_start,record_end
0,1,BREST,1807,2019
1,10,SAN FRANCISCO,1855,2020
2,100,PORTO CORSINI,1970,1972
3,1000,TUKTOYAKTUK,1962,2019
4,1001,SAN JUAN,1963,2020
5,1002,TRIBENI,1962,2006
6,1003,HONNGU,1962,2013
7,1004,WESTPORT HARBOUR,1985,1993


## 6. Pilot: final windows only

This step estimates the 1960–2019 and 1950–2019 windows under both completeness settings. With the previously analyzed dataset, the 70-year window had **133 eligible / 87 selected** stations at 75% completeness and **112 eligible / 75 selected** at 87% completeness. These are dataset-specific reference counts, not constraints imposed on the calculation.

The pilot uses the configured number of draws and the same per-window random seeds as the full run. Its point estimates therefore agree with the corresponding full-run estimates. The elapsed time provides an approximate indication of full-run cost.


In [9]:
if list(CONFIG['year_limits']) != LOADED_YEAR_LIMITS:
    raise RuntimeError('The analysis period changed; rerun Section 5.')
pilot_started = time.perf_counter()
pilot_fits = fit_windows(records, CONFIG, final_only=True)
pilot_summary, pilot_draws, pilot_membership = estimate_windows(
    pilot_fits, CONFIG, final_only=True)
assert_results(pilot_summary, CONFIG)
print(f'Pilot elapsed time: {time.perf_counter()-pilot_started:.1f} s')
display(pilot_summary[['window_length', 'completeness', 'start_year', 'end_year',
                       'gamma', 'eligible', 'selected', 'n_retained', 'sample_size',
                       'mean_selected', 'median_selected', 'resampled_mean',
                       'resampled_median_mean', 'M_prime', 'median_subsample_sd']]
        .round(6))


Fitted 60-year window 1960–2019
Fitted 70-year window 1950–2019
Estimated 60-year windows; completeness=0.75
Estimated 60-year windows; completeness=0.87
Estimated 70-year windows; completeness=0.75
Estimated 70-year windows; completeness=0.87
Pilot elapsed time: 1.0 s


,window_length,completeness,start_year,end_year,gamma,eligible,selected,n_retained,sample_size,mean_selected,median_selected,resampled_mean,resampled_median_mean,M_prime,median_subsample_sd
0,60,0.75,1960,2019,0.00,181,127,127,64,3.052270,2.380723,3.054045,2.438365,2.380970,0.193534
1,60,0.75,1960,2019,0.10,181,127,114,57,3.052270,2.380723,2.452088,2.251771,2.243932,0.116236
2,60,0.75,1960,2019,0.25,181,127,95,48,3.052270,2.380723,2.135039,2.110221,2.100678,0.060103
3,60,0.87,1960,2019,0.00,145,103,103,52,3.002629,2.343408,3.006413,2.420487,2.362312,0.191859
4,60,0.87,1960,2019,0.10,145,103,93,47,3.002629,2.343408,2.449307,2.261563,2.244087,0.117374
5,60,0.87,1960,2019,0.25,145,103,77,39,3.002629,2.343408,2.110946,2.113084,2.103329,0.073395
6,70,0.75,1950,2019,0.00,133,87,87,44,2.905477,2.484035,2.909977,2.391295,2.453759,0.249071
7,70,0.75,1950,2019,0.10,133,87,78,39,2.905477,2.484035,2.270212,2.165955,2.086445,0.217493
8,70,0.75,1950,2019,0.25,133,87,65,33,2.905477,2.484035,1.956353,1.943997,1.957339,0.101664
9,70,0.87,1950,2019,0.00,112,75,75,38,2.871673,2.125775,2.877469,2.289505,2.254964,0.281253


## 7. Full calculation

Set `RUN_FULL = True` and execute this cell after inspecting the pilot. The defaults produce 61 windows of length 60 and 51 windows of length 70, each with two completeness settings and three trimming fractions. Progress messages report completed windows and configurations. Regression fits are reused across completeness settings.

This is the most time-consuming cell. The fresh Python subsampling stream is deterministic for each configuration and window. Changing the order of the settings or adding another trimming fraction leaves existing point-estimate streams unchanged. Rerunning this cell replaces the in-memory full results. Saved export folders are created only in Section 12.


In [10]:
RUN_FULL = False

if RUN_FULL:
    if list(CONFIG['year_limits']) != LOADED_YEAR_LIMITS:
        raise RuntimeError('The analysis period changed; rerun Section 5.')
    validate_config(CONFIG)
    RUN_CONFIG = json.loads(json.dumps(CONFIG))
    RUN_SNAPSHOT_NOTE = DATA_SNAPSHOT_NOTE
    full_started = time.perf_counter()
    all_fits = fit_windows(records, RUN_CONFIG)
    all_estimates, all_draws, all_membership = estimate_windows(all_fits, RUN_CONFIG)
    assert_results(all_estimates, RUN_CONFIG)
    print(f'Full calculation completed in {time.perf_counter()-full_started:.1f} s')
    print('Summary rows:', len(all_estimates))
    print('Undefined estimates:', int(all_estimates.M_prime.isna().sum()))
else:
    print('Full calculation is off. Set RUN_FULL = True after the pilot check.')


Fitted 60-year window 1900–1959
Fitted 60-year window 1909–1968
Fitted 60-year window 1919–1978
Fitted 60-year window 1929–1988
Fitted 60-year window 1939–1998
Fitted 60-year window 1949–2008
Fitted 60-year window 1959–2018
Fitted 60-year window 1960–2019
Fitted 70-year window 1900–1969
Fitted 70-year window 1909–1978
Fitted 70-year window 1919–1988
Fitted 70-year window 1929–1998
Fitted 70-year window 1939–2008
Fitted 70-year window 1949–2018
Fitted 70-year window 1950–2019
Estimated 60-year windows; completeness=0.75
Estimated 60-year windows; completeness=0.87
Estimated 70-year windows; completeness=0.75
Estimated 70-year windows; completeness=0.87
Full calculation completed in 58.0 s
Summary rows: 672
Undefined estimates: 0


## 8. Inspect the results and sampling precision

The exact finite-population SD provides a reference for the Monte Carlo SD of the subsample means. The diagnostic ratio should fluctuate around 1 across windows; it is a numerical check of the subsampling calculation. The final table distinguishes direct summaries from their resampling averages and the final $M'$ estimator.


In [11]:
if 'all_estimates' not in globals():
    raise RuntimeError('Complete Section 7 with RUN_FULL = True before inspecting full results.')

counts = all_estimates[all_estimates.gamma == 0]
display(counts.groupby(['window_length', 'completeness']).agg(
    n_windows=('start_year', 'size'), eligible_min=('eligible', 'min'),
    eligible_max=('eligible', 'max'), selected_min=('selected', 'min'),
    selected_max=('selected', 'max'), selected_station_windows=('selected', 'sum')))

final_table = all_estimates[all_estimates.end_year == RUN_CONFIG['year_limits'][1]]
display(final_table[['window_length', 'completeness', 'gamma', 'selected',
                     'n_retained', 'sample_size', 'mean_selected', 'median_selected',
                     'resampled_mean', 'resampled_median_mean', 'median_retained',
                     'M_prime', 'mean_subsample_sd', 'median_subsample_sd',
                     'nominal_99_halfwidth']].round(6))

nondegenerate = all_estimates.mean_subsample_sd_exact > 0
sd_ratio = (all_estimates.loc[nondegenerate, 'mean_subsample_sd'] /
            all_estimates.loc[nondegenerate, 'mean_subsample_sd_exact'])
print('Monte Carlo / exact subsample-mean SD:')
display(sd_ratio.quantile([0.05, 0.50, 0.95]).rename('ratio'))
display(final_table.loc[final_table.gamma == 0.25,
                       ['window_length', 'completeness', 'M_prime',
                        'nominal_99_halfwidth', 'envelope_valid_windows']].round(6))


n_windows  eligible_min  eligible_max  \
window_length completeness                                          
60            0.75                 61            55           202   
              0.87                 61            46           157   
70            0.75                 51            52           149   
              0.87                 51            47           129   

                            selected_min  selected_max  \
window_length completeness                               
60            0.75                    31           143   
              0.87                    26           114   
70            0.75                    31           103   
              0.87                    28            91   

                            selected_station_windows  
window_length completeness                            
60            0.75                              4327  
              0.87                              3536  
70            0.75                              3100  
              0.87                              2766

,window_length,completeness,gamma,selected,n_retained,sample_size,mean_selected,median_selected,resampled_mean,resampled_median_mean,median_retained,M_prime,mean_subsample_sd,median_subsample_sd,nominal_99_halfwidth
180,60,0.75,0.00,127,127,64,3.052270,2.380723,3.054045,2.438365,2.380723,2.380970,0.212210,0.193534,0.240739
181,60,0.75,0.10,127,114,57,3.052270,2.380723,2.452088,2.251771,2.244009,2.243932,0.094183,0.116236,0.150601
182,60,0.75,0.25,127,95,48,3.052270,2.380723,2.135039,2.110221,2.098027,2.100678,0.071804,0.060103,0.075581
363,60,0.87,0.00,103,103,52,3.002629,2.343408,3.006413,2.420487,2.343408,2.362312,0.228139,0.191859,0.246372
364,60,0.87,0.10,103,93,47,3.002629,2.343408,2.449307,2.261563,2.244087,2.244087,0.103348,0.117374,0.145498
365,60,0.87,0.25,103,77,39,3.002629,2.343408,2.110946,2.113084,2.103329,2.103329,0.083472,0.073395,0.090212
516,70,0.75,0.00,87,87,44,2.905477,2.484035,2.909977,2.391295,2.484035,2.453759,0.260681,0.249071,0.318515
517,70,0.75,0.10,87,78,39,2.905477,2.484035,2.270212,2.165955,2.054884,2.086445,0.111688,0.217493,0.275957
518,70,0.75,0.25,87,65,33,2.905477,2.484035,1.956353,1.943997,1.957339,1.957339,0.096936,0.101664,0.135969
669,70,0.87,0.00,75,75,38,2.871673,2.125775,2.877469,2.289505,2.125775,2.254964,0.286518,0.281253,0.362545


Monte Carlo / exact subsample-mean SD:


0.05    0.965560
0.50    0.999070
0.95    1.033215
Name: ratio, dtype: float64

,window_length,completeness,M_prime,nominal_99_halfwidth,envelope_valid_windows
182,60,0.75,2.100678,0.075581,61
365,60,0.87,2.103329,0.090212,61
518,70,0.75,1.957339,0.135969,51
671,70,0.87,1.926253,0.140818,51


## 9. Plotting and export functions

All figures are created once and displayed with `plt.show()`. Cells avoid returning a figure object as their final expression, preventing duplicate notebook displays.


In [ ]:
def plot_estimator_comparison(summary, cfg, gamma=0.25):
    """Mean, median, and final trimmed estimates with distinct dispersion measures."""
    figures = {}
    if gamma not in cfg['gammas']:
        raise ValueError('The displayed trimming fraction must be present in gammas.')
    colors = plt.get_cmap('tab10').colors
    for fraction in cfg['completeness_values']:
        fig, axes = plt.subplots(1, 3, figsize=(15, 4.3), sharex=True)
        for k, w in enumerate(cfg['window_lengths']):
            color = colors[k % len(colors)]
            base = summary[(summary.window_length == w) &
                           (summary.completeness == fraction) & (summary.gamma == 0)].sort_values('start_year')
            trimmed = summary[(summary.window_length == w) &
                              (summary.completeness == fraction) & (summary.gamma == gamma)].sort_values('start_year')
            x = base.window_label.to_numpy(float)
            for ax, direct, center, sd in (
                (axes[0], 'mean_selected', 'resampled_mean', 'mean_subsample_sd'),
                (axes[1], 'median_selected', 'resampled_median_mean', 'median_subsample_sd')):
                y, spread = base[center].to_numpy(float), base[sd].to_numpy(float)
                ax.plot(x, base[direct], color=color, linestyle='--', linewidth=1.3,
                        label=f'{w} yr: direct')
                ax.plot(x, y, color=color, linewidth=1.7, label=f'{w} yr: resampled average')
                ax.fill_between(x, y-spread, y+spread, color=color, alpha=0.14)
            y = trimmed.M_prime.to_numpy(float)
            spread = trimmed.nominal_99_halfwidth.to_numpy(float)
            axes[2].plot(trimmed.window_label, y, color=color, label=f'{w} yr', linewidth=1.7)
            if np.isfinite(spread).any():
                axes[2].fill_between(trimmed.window_label, y-spread, y+spread, color=color, alpha=0.14)
        envelope_note = 'nominal 99% envelope' if cfg['n_resamples'] == 1000 else 'point estimates'
        titles = ['(a) Mean: ±1 subsampling SD', '(b) Median: ±1 subsampling SD',
                  rf'(c) $M^\prime$, $\gamma={gamma:g}$: {envelope_note}']
        for ax, title in zip(axes, titles):
            ax.set_title(title, fontsize=10)
            ax.set_xlabel('Window label (year)')
            ax.set_ylabel('Rate (mm/yr)')
            ax.grid(linestyle=':', alpha=0.5)
            ax.legend(fontsize=8, framealpha=0.9)
        fig.suptitle(f'Minimum completeness: {100*fraction:g}%')
        fig.tight_layout()
        figures[f'estimators_completeness_{round(100*fraction)}'] = fig
    return figures


def plot_trimming(summary, cfg, envelope_gamma=0.25):
    """Compare the prescribed trimming fractions within each window length."""
    figures = {}
    for fraction in cfg['completeness_values']:
        fig, axes = plt.subplots(1, len(cfg['window_lengths']),
                                 figsize=(6.5*len(cfg['window_lengths']), 4.4), squeeze=False)
        for ax, w in zip(axes[0], cfg['window_lengths']):
            for gamma in cfg['gammas']:
                d = summary[(summary.window_length == w) & (summary.completeness == fraction)
                            & (summary.gamma == gamma)].sort_values('start_year')
                x, y = d.window_label.to_numpy(float), d.M_prime.to_numpy(float)
                line, = ax.plot(x, y, label=rf'$M^\prime$: $\gamma={gamma:g}$', linewidth=1.7)
                if gamma == envelope_gamma:
                    band = d.nominal_99_halfwidth.to_numpy(float)
                    if np.isfinite(band).any():
                        ax.fill_between(x, y-band, y+band, color=line.get_color(), alpha=0.15)
                if gamma == 0:
                    ax.plot(x, d.median_selected, color='0.35', linestyle=':', linewidth=1.1,
                            label='Direct untrimmed median')
            ax.set_title(f'{w}-year windows')
            ax.set_xlabel('Window label (year)')
            ax.set_ylabel('Rate (mm/yr)')
            ax.grid(linestyle=':', alpha=0.5)
            ax.legend(fontsize=9)
        fig.suptitle(f'Trimming; minimum completeness {100*fraction:g}%')
        fig.tight_layout()
        figures[f'trimming_completeness_{round(100*fraction)}'] = fig
    return figures


def plot_station_counts(summary, cfg):
    fig, axes = plt.subplots(1, len(cfg['window_lengths']),
                             figsize=(6.5*len(cfg['window_lengths']), 4.2), squeeze=False)
    for ax, w in zip(axes[0], cfg['window_lengths']):
        for fraction in cfg['completeness_values']:
            d = summary[(summary.window_length == w) & (summary.completeness == fraction)
                        & (summary.gamma == 0)].sort_values('start_year')
            line, = ax.plot(d.window_label, d.selected, label=f'{100*fraction:g}%: selected')
            ax.plot(d.window_label, d.eligible, color=line.get_color(), linestyle='--',
                    label=f'{100*fraction:g}%: eligible')
        ax.set_title(f'{w}-year windows')
        ax.set_xlabel('Window label (year)')
        ax.set_ylabel('Number of stations')
        ax.grid(linestyle=':', alpha=0.5)
        ax.legend(fontsize=9)
    fig.tight_layout()
    return {'station_counts': fig}


def export_results(output_root, cfg, fits, summary, draws, membership,
                   manifest, manifest_hash, snapshot_note, figures):
    """Save tables, resampling arrays, figure files, and reproducibility metadata."""
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    folder = Path(output_root)/f'run_{stamp}'
    folder.mkdir(parents=True, exist_ok=False)
    fits.to_csv(folder/'station_window_fits.csv', index=False, float_format='%.12g')
    summary.to_csv(folder/'window_estimates.csv', index=False, float_format='%.12g')
    membership.to_csv(folder/'selected_station_membership.csv', index=False, float_format='%.12g')
    final = summary[summary.end_year == cfg['year_limits'][1]]
    final.to_csv(folder/'final_window_estimates.csv', index=False, float_format='%.12g')
    arrays, array_index = {}, []
    for i, ((w, completeness, gamma), values) in enumerate(draws.items()):
        prefix = f'case_{i:03d}'
        for field, value in values.items():
            arrays[f'{prefix}_{field}'] = value
        array_index.append(dict(prefix=prefix, window_length=w,
                                completeness=completeness, gamma=gamma))
    np.savez_compressed(folder/'subsampling_draws.npz', **arrays)
    (folder/'subsampling_array_index.json').write_text(json.dumps(array_index, indent=2)+'\n')
    (folder/'input_manifest.json').write_text(json.dumps(manifest, indent=2)+'\n')
    metadata = dict(
        implementation_version='1.0.0', created_utc=stamp, config=cfg,
        data_snapshot_note=snapshot_note, input_manifest_sha256=manifest_hash,
        units=dict(annual_height='mm', slope='mm/yr', subsampling_dispersion='mm/yr'),
        selection='OLS slope > 0 and two-sided p <= pcrit_two_sided',
        record_span='Raw first/last station years must contain the entire window',
        quality_flag_policy='Parsed; no additional flag-based exclusions',
        rounding='floor(value + 0.5) for retained and subsample station counts',
        sampling='Independent subsets across repetitions; without replacement within a subset',
        generator='NumPy default_rng / PCG64; SeedSequence keyed by configuration and window',
        median_comparison='Arithmetic average of subsample medians',
        final_estimator='Median of subsample medians after upper-tail trimming',
        window_label='start_year + window_length/2; inclusive end_year=start_year+window_length-1',
        envelope='s_j / sqrt(sum_k s_k^2) times the 991st of 1000 ordered trajectory norms',
        envelope_enabled=bool(cfg['n_resamples'] == 1000),
        uncertainty_scope='Conditional station subsampling; overlapping-window covariance is not preserved',
        versions=dict(python=platform.python_version(), numpy=np.__version__,
                      scipy=scipy.__version__, pandas=pd.__version__, matplotlib=matplotlib.__version__),
    )
    (folder/'metadata.json').write_text(json.dumps(metadata, indent=2)+'\n')
    for name, fig in figures.items():
        fig.savefig(folder/f'{name}.jpg', dpi=300, bbox_inches='tight')
        fig.savefig(folder/f'{name}.pdf', bbox_inches='tight')
    (folder/'README.txt').write_text(EXPORT_README, encoding='utf-8')
    return folder


EXPORT_README = '''Annual tide-gauge summaries

station_window_fits.csv: one row per fitted station-window, meeting the least
restrictive configured completeness threshold. p-values are ordinary OLS.
For a stricter setting, filter observed_fraction >= completeness before applying
positive_significant. Raw record spans must contain the full observation window.

window_estimates.csv: one row per window, completeness, and trimming fraction.
mean_selected / median_selected: direct summaries before trimming.
mean_retained / median_retained: direct summaries after trimming.
resampled_mean: average of subsample means.
resampled_median_mean: average of subsample medians (median-comparison figure).
M_prime: median of subsample medians (final trimmed estimator).
mean_subsample_sd / median_subsample_sd: sample SD across the resampled statistics.
mean_subsample_sd_exact: finite-population conditional SD of subsample means.
nominal_99_halfwidth: legacy descriptive envelope, computed across all valid
windows for each configuration, using the 991st ordered norm of 1000 repetitions.
The envelope is conditional on the selected station population and does not
preserve covariance between overlapping windows. With other repetition counts
the envelope is left undefined. Missing populations also produce NaN estimates.

selected_station_membership.csv: station identities, slopes, ranks, and retained
flags for every trimming fraction. Tied slopes are ordered by station ID.
final_window_estimates.csv: subset ending at the configured final year.
subsampling_draws.npz: all subsample means and medians; arrays have shape
(number of windows, number of repetitions). The associated JSON file maps
case prefixes to window length, completeness, and trimming fraction.
metadata.json: configuration, definitions, seed scheme, and dependency versions.
input_manifest.json: SHA-256 hashes of the input catalogue and station files.
JPG/PDF files: figures displayed by the notebook.

All rate quantities are in mm/yr. Station counts precede trimming unless named
n_retained or sample_size. The baseline procedure uses annual OLS inference;
HAC/BH sensitivities, fixed-cohort analysis, and stochastic benchmarks are separate
analyses. Independent uniform subsets are used here. The older MATLAB helper
attempted to reject repeated subsets, so archived random realizations can differ.
The mean, median, and trimmed summaries describe the positively selected local
station trends. Their subsampling variability is conditional on those records.
'''


## 10. Mean, median, and trimmed estimate

For each completeness setting, panels show:

- **Mean:** the direct selected-station mean (dashed), average of subsample means (solid), and ±1 subsampling SD.
- **Median:** the direct selected-station median (dashed), average of subsample medians (solid), and ±1 subsampling SD.
- **Trimmed estimate:** $M'$ at $\gamma=0.25$, with the nominal 99% descriptive envelope.

The different uncertainty measures are identified in the panel titles. No extrapolation beyond the available full windows is performed. `PLOT_GAMMA` may be set to another fraction present in the full-run configuration.


In [ ]:
PLOT_GAMMA = 0.25
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})
comparison_figures = plot_estimator_comparison(all_estimates, RUN_CONFIG, gamma=PLOT_GAMMA)
plt.show()


## 11. Trimming sensitivity and station counts

The trimming figure displays $M'$ for every configured $\gamma$, alongside the direct untrimmed median as a dotted reference. The $\gamma=0$ resampled estimator is the median of subsample medians, so it may differ slightly from that reference. Shading marks the nominal envelope for `PLOT_GAMMA`.

The station-count figure separates Stage 1 eligibility from positive-significance selection. Counts precede upper-tail trimming. Changes in these populations accompany the sliding-window evolution of the estimates.


In [ ]:
trimming_figures = plot_trimming(all_estimates, RUN_CONFIG, envelope_gamma=PLOT_GAMMA)
plt.show()
count_figures = plot_station_counts(all_estimates, RUN_CONFIG)
plt.show()


## 12. Export reproducible results

The export creates a new timestamped folder containing:

- station-level fitted slopes, OLS p-values, and completeness information;
- window estimates, counts, SDs, and nominal envelopes;
- selected station identities, trimming ranks, and retained/removed flags;
- all subsample means and medians in a compressed NumPy archive;
- figure files in JPG and PDF formats;
- configuration, dependency versions, input-file hashes, and a field-description README.

Files are written with unrounded numerical results. `SAVE_RESULTS = False` skips writing. The notebook can be distributed with a separate archived PSMSL data snapshot or an appropriate dataset reference; the manifest identifies the exact inputs used in a completed run.


In [ ]:
SAVE_RESULTS = True

if SAVE_RESULTS:
    figures_to_save = {**comparison_figures, **trimming_figures, **count_figures}
    export_folder = export_results(
        OUTPUT_DIR, RUN_CONFIG, all_fits, all_estimates, all_draws, all_membership,
        INPUT_MANIFEST, INPUT_FINGERPRINT, RUN_SNAPSHOT_NOTE, figures_to_save)
    print('Saved results:', export_folder.resolve())
    print('Saved files:')
    for path in sorted(export_folder.iterdir()):
        print(' ', path.name)
else:
    print('File export is off.')


## 13. Interpretation and reproducibility notes

The calculated rates summarize the selected local relative-sea-level trends. The positive-significance selection, the choice of statistic, and upper-tail trimming each influence the reported values. The subsampling SDs and nominal envelopes are conditional on the observed station sets and their fitted slopes.

Overlapping windows share observations, and station membership can change. Visual smoothness and changes in adjacent estimates are descriptive features of these summaries. The fixed-cohort and HAC/BH notebooks provide the corresponding sensitivity analyses.

The use of annual data and conventional OLS selection is stated explicitly in this baseline implementation. Annual averaging does not itself establish temporal independence. VLM corrections, spatial weighting, or spatially dependent resampling are not applied here.

For an archival release, retain this notebook, the input-data snapshot identifier, exported metadata and manifests, and the resulting figures and tables. The saved package versions and deterministic seeds document the numerical environment. Raw PSMSL files are read only; all generated files go into a new results folder.
